# 02 — Sentiment Analysis

Evaluate the XLM-RoBERTa model on fixture tweets and compare with VADER fallback.

In [ ]:
import json, sys
from pathlib import Path
sys.path.insert(0, str(Path("..") / "src"))
from ph_sentiment.processor.sentiment import classify_batch, classify_single
from ph_sentiment.processor.enrichment import enrich_batch
from ph_sentiment.producer.simulator import parse_tweets


## Load and classify tweets

In [ ]:
raw = json.load(open("fixtures/WeLoveTheEarth.json"))
tweets = parse_tweets(raw)
tweets = enrich_batch(tweets)

# Assign a topic
for t in tweets:
    t.topic_name = "#WeLoveTheEarth"

classified = classify_batch(tweets)
print(f"Classified {len(classified)} tweets")


## Results breakdown

In [ ]:
import pandas as pd
results = pd.DataFrame([{
    "lang": t.lang,
    "text": t.text[:80],
    "label": t.sentiment_label,
    "score": t.sentiment_score,
    "hashtags": t.hashtags,
} for t in classified])

print(results[["lang","label","score"]].value_counts().to_string())
print(f"\nAvg sentiment score: {results['score'].mean():.4f}")
print(f"Positive: {(results['label']=='positive').mean()*100:.1f}%")
print(f"Neutral:  {(results['label']=='neutral').mean()*100:.1f}%")
print(f"Negative: {(results['label']=='negative').mean()*100:.1f}%")


## Taglish examples

In [ ]:
taglish = results[results["lang"] == "tl"]
print("Taglish tweet classifications:")
for _, row in taglish.iterrows():
    print(f"  [{row['label']:8s} {row['score']:+.2f}] {row['text']}")


## Single-text classification demo

In [ ]:
test_texts = [
    "Mahal ko ang ating kalikasan! Ingatan natin ang mundo.",
    "Grabe yung pollution dito. Nakakainis talaga.",
    "The weather is okay today.",
]
print("Single-text classifications:")
for text in test_texts:
    label, score = classify_single(text)
    print(f"  [{label:8s} {score:+.2f}] {text[:60]}")
